# Stage 0. Extract fields of view

## 1. Setup

### 1.1 Imports

In [ ]:
import datetime
import logging
import os
from pathlib import Path

import numpy as np
import pandas as pd
from ome_types import to_xml
from omegaconf import OmegaConf

from acid.config import load_config, prepare_output
from acid.image_processing.extract_metadata import extract_bioio_scene_metadata
from acid.image_processing.make_imagej_metadata import imagej_compatible_metadata_dict
from acid.image_processing.name_metadata import extract_name_metadata
from acid.image_processing.save_metadata import save_xml_string
from acid.utils.listdirNHF import listdirNHF
from acid.utils.open_image import bioio_open_image
from acid.utils.save_image import tifffile_save_ometiff

### 1.2 Configure logging

In [ ]:
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

### 1.3 Load configuration

In [ ]:
CONFIG_FILE = None
config = load_config(CONFIG_FILE)
prepare_output(config)
stage_cfg = config.field_of_view_extraction
paths_cfg = config.shared.paths

## 2. Input Data Preparation

In [ ]:
input_directory = Path(stage_cfg.acquisitions.directory)
if not input_directory.is_dir():
    raise FileNotFoundError(f"Raw acquisition directory not found: {input_directory}")
Path(stage_cfg.image_saving.directory).mkdir(parents=True, exist_ok=True)
Path(stage_cfg.metadata.directory).mkdir(parents=True, exist_ok=True)

## 3. Field-of-view extraction
### 3.1 Helpers

In [ ]:
# FUTURE: move to src/acid/io/acquisition_discovery.py
def discover_acquisitions(cfg):
    root = Path(cfg.acquisitions.directory)
    selection = OmegaConf.to_container(
        cfg.acquisitions.experiment_directory_selection, resolve=True
    )
    file_selection = OmegaConf.to_container(
        cfg.acquisitions.file_selection, resolve=True
    )
    acquisitions = []
    for name in sorted(
        listdirNHF(root, target=selection["include"], exclude=selection["exclude"])
    ):
        directory = root / name
        if not directory.is_dir():
            continue
        for filename in sorted(
            listdirNHF(
                directory,
                target=file_selection["include"],
                exclude=file_selection["exclude"],
            )
        ):
            path = directory / filename
            if path.is_file():
                acquisitions.append(path)
    if not acquisitions:
        raise ValueError(f"No matching acquisition files in {root}")
    return acquisitions


# FUTURE: move to src/acid/image_processing/extract_field_of_view.py
def extract_acquisition(acquisition, cfg):
    input_file = acquisition.name
    experiment_subdir = acquisition.parent.name
    scenes_metadata_collection = []
    print("=========")
    print(f"working on {input_file}")
    bioio_input_image, input_image_metadata = bioio_open_image(
        str(acquisition), return_metadata=True
    )
    xml_input_image_metadata = to_xml(input_image_metadata)
    save_xml_string(
        xml_input_image_metadata,
        os.path.join(
            cfg.image_saving.directory,
            f"{acquisition.stem}{cfg.image_saving.save_file_name_separator}{cfg.image_saving.xml_suffix}",
        ),
    )
    for scene_n, scene in enumerate(bioio_input_image.scenes):
        print("---------")
        print(f"working on scene {scene}")
        bioio_input_image.set_scene(scene)
        experiment = experiment_subdir.split(cfg.naming.experiment_separator)[
            cfg.naming.experiment_index
        ]
        name_metadata_dict = extract_name_metadata(
            acquisition.stem,
            separator=cfg.naming.acquisition_filename_separator,
            infobits={
                "processing_date": datetime.datetime.now()
                .astimezone()
                .strftime(cfg.image_metadata.processing_date_format),
                "experiment": experiment,
                "condition_1": tuple(cfg.naming.condition_1_bitinfo),
                "infectious_organism": tuple(cfg.naming.infectious_organism_bitinfo),
                "condition_2": tuple(cfg.naming.condition_2_bitinfo),
                "imaging_hours": tuple(cfg.naming.imaging_hours_bitinfo),
                "well": tuple(cfg.naming.well_bitinfo),
            },
        )
        save_file_name = f"{acquisition.stem}{cfg.naming.acquisition_filename_separator}{experiment.replace(cfg.naming.sub_experiment_separator, cfg.naming.sub_experiment_separator_replacement)}{cfg.naming.acquisition_filename_separator}{scene}{cfg.image_saving.ome_suffix}"
        scene_metadata_series, scene_metadata_dict = extract_bioio_scene_metadata(
            bioio_scene=bioio_input_image,
            dims_order_name=cfg.image_metadata.dimensions_order_metadata_name,
            raw_file_name=input_file,
            scene_name=scene,
            processing_date_yymmdd=name_metadata_dict["processing_date"],
            ome_tif_file_name=save_file_name,
            location=cfg.image_metadata.location,
            microscope=cfg.image_metadata.microscope,
            objective=cfg.image_metadata.objective,
            experiment=experiment,
            condition_1=name_metadata_dict["condition_1"],
            infectious_organism=name_metadata_dict["infectious_organism"],
            condition_2=name_metadata_dict["condition_2"],
            imaging_hours=name_metadata_dict["imaging_hours"],
            well=name_metadata_dict["well"],
            channel_0=cfg.image_metadata.channels[0],
            channel_1=cfg.image_metadata.channels[1],
            channel_2=cfg.image_metadata.channels[2],
            channel_3=cfg.image_metadata.channels[3],
            channel_4=cfg.image_metadata.channels[4],
            physical_size_unit_x=cfg.image_metadata.physical_size_unit,
            physical_size_unit_y=cfg.image_metadata.physical_size_unit,
        )
        scenes_metadata_collection.append(scene_metadata_series)
        input_scene = bioio_input_image.data
        input_scene = np.squeeze(input_scene)
        imagej_scene_metadata_dict = imagej_compatible_metadata_dict(
            scene_metadata_dict
        )
        tifffile_save_ometiff(
            os.path.join(cfg.image_saving.directory, save_file_name),
            data=input_scene,
            imagej=cfg.image_saving.save_imagej_compatible,
            photometric=cfg.image_saving.photometric,
            metadata=imagej_scene_metadata_dict,
        )
    return scenes_metadata_collection


# FUTURE: move to src/acid/image_processing/extract_field_of_view.py
def extract_fields_of_view(acquisitions, cfg):
    if len(cfg.image_metadata.channels) != 5:
        raise ValueError("This extraction workflow requires five configured channels")
    scenes = []
    for acquisition in acquisitions:
        logging.getLogger(__name__).info("Extracting %s", acquisition)
        scenes.extend(extract_acquisition(acquisition, cfg))
    if not scenes:
        raise ValueError("No fields of view were extracted")
    return pd.concat(scenes, axis=1).T

### 3.2 Extract acquisitions

In [ ]:
acquisitions = discover_acquisitions(stage_cfg)
metadata_df = extract_fields_of_view(acquisitions, stage_cfg)

## 4. Save metadata

In [ ]:
# FUTURE: move to src/acid/utils/metadata/saving.py
def save_metadata_dataframe(metadata_df, metadata_config, project_name):
    saving = metadata_config.saving
    separator = saving.save_file_name_separator
    suffix = saving.metadata_file_suffix.format(save_file_name_separator=separator)
    filename = separator.join(
        [
            datetime.datetime.now().astimezone().strftime(saving.metadata_date_format),
            project_name,
            saving.metadata_savingword,
            suffix,
        ]
    )
    path = Path(metadata_config.directory) / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    metadata_df.to_csv(path, index=saving.save_csv_index)
    logging.getLogger(__name__).info("Saved metadata: %s", path)
    return path


metadata_path = save_metadata_dataframe(
    metadata_df, stage_cfg.metadata, config.project_identity.project_name
)

## 5. Inspect results

In [ ]:
metadata_df.head()